## Json Event Flattening



In [0]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
from pyspark.sql.types import *


In [0]:
def extract(input_path:str) -> DataFrame:
    """ Read the JSON Lines source file from input_path. """
    df = spark.read.format('json').option("multiLine",True).load(input_path)
    return df


In [0]:

def transform(df: DataFrame) -> DataFrame:
    """Flatten nested metadata fields into top-level columns."""
    df = df.select("user_id","event_type","timestamp",F.col("metadata.device"),"metadata.os","metadata.browser").drop("metadata")
    return df 


In [0]:

def load(df: DataFrame, output_path: str):
    """Write the flattened DataFrame as Parquet to output_path."""
    pass

In [0]:
events_path = "/Volumes/spark_data/dev/spark_data_volume/manish_etl/events.json"
df = extract(input_path=events_path)
display(df)




In [0]:
df = transform(df)
df.show()

In [0]:
json_data = [
    {
        "id": 1,
        "info": {"name": "Alice", "city": "New York"},
        "orders": [{"item": "Laptop", "qty": 1}, {"item": "Mouse", "qty": 2}]
    },
    {
        "id":2,
        "info": {'name':"Alex","city":"Texas"},
        "orders":[{'item':"Mobile","qty":1},{"item":"Phone Stand","qty":4}]
    }
]

df = spark.createDataFrame(json_data)
df.show()


In [0]:
df_struct = df.select("id","info.name","info.city","orders")
# display(df_struct)
df_flatten = df_struct.withColumn("orders", F.explode("orders")).select("id","name","city","orders.item","orders.qty")
display(df_flatten)

In [0]:
def flatten_json(df: DataFrame) -> DataFrame:
    """
    Recursively flattens all StructTypes and ArrayTypes in a PySpark DataFrame.
    """
    # Identify all complex fields (Structs and Arrays)
    complex_fields = [
        field for field in df.schema.fields 
        if isinstance(field.dataType, (StructType, ArrayType))
    ]
    
    # If no complex fields remain, return the flat DataFrame
    if not complex_fields:
        return df

    for field in complex_fields:
        col_name = field.name
        data_type = field.dataType

        # Handle StructType (Objects)
        if isinstance(data_type, StructType):
            # Expand all sub-fields using dot notation
            expanded = [F.col(f"{col_name}.{sub_field.name}").alias(f"{col_name}_{sub_field.name}") for sub_field in data_type.fields]
            df = df.select("*", *expanded).drop(col_name)

        # Handle ArrayType (Lists)
        elif isinstance(data_type, ArrayType):
            # Explode the array into rows. explode_outer preserves null rows.
            df = df.withColumn(col_name, F.explode_outer(col_name))

    # Recursively call the function until all layers are flat
    return flatten_json(df)

# Usage:
flat_df = flatten_json(df)
display(flat_df)

## Schema Evolution

In [0]:
%sql
create table if not exists spark_data.dev.test_customers
(
    id int,
    name string
);

insert into spark_data.dev.test_customers
values(1,'alex'),(2,'mani');

In [0]:
%sql
alter table spark_data.dev.test_customers
add column country string;

update spark_data.dev.test_customers
set country='poland';

In [0]:
%sql
update spark_data.dev.test_customers
set name='Alex'

In [0]:
%sql
create or replace view spark_data.dev.v_evolutioncust
with schema evolution
as
select * from spark_data.dev.test_customers

In [0]:
%sql
select * from spark_data.dev.v_defaultcust;


In [0]:
%sql
select * from spark_data.dev.v_evolutioncust;

## CSV to Data Warehouse


In [0]:
user_path = "/Volumes/spark_data/dev/spark_data_volume/manish_etl/users.csv"

users_parquet_path = "/Volumes/spark_data/dev/spark_data_volume/manish_etl/users_pqt"
users_df = (
            spark.read
            .format('csv')
            .option('header','true')
            .load(user_path)
            )

display(users_df)

In [0]:

def extract(spark, input_path: str) -> DataFrame:
    """Read the CSV source file from input_path."""
    users_df = (
        spark.read.format('csv')
                  .option('header','true')
                  .load(input_path)
    )
    return users_df

def transform(df: DataFrame) -> DataFrame:
    """Clean: trim names, cast age to int, filter age >= 18, uppercase names."""
    clean_df = (    df.withColumn('age',F.col('age').try_cast('int'))
                      .withColumn('name',F.trim('name'))
                      .filter((F.col('age')>=18) & (F.col('age').isNotNull()))
                      .select(F.upper('name'),"age","city","signup_date")
                      
                )
    return clean_df

def load(df: DataFrame, output_path: str):
    """Write the cleaned DataFrame as Parquet to output_path."""
    (
        df.write.format('parquet').mode('overwrite').save(output_path)
    )


In [0]:
df = extract(spark,user_path)

trans_df = transform(df)

load(trans_df,users_parquet_path)
